# FASE 2A: Modelado Clásico

**Proyecto Integrador — Grupo 2:** Análisis de Satisfacción en Productos de Oficina
**Modelos:** Regresión Logística, Random Forest, LightGBM, XGBoost

In [15]:
# Instalacion de dependencias
!pip install mlflow lazypredict -q

## Configuración del Entorno

In [2]:
import os
import sys
import gc
import json
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import mlflow

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score, precision_recall_fscore_support
from sklearn.utils.class_weight import compute_class_weight

import lightgbm as lgb
import xgboost as xgb

from scipy.sparse import hstack, csr_matrix
from tqdm.auto import tqdm

RANDOM_SEED = 42

FEATURE_COLS = ['mayusculas_count', 'char_total', 'exclamacion_count',
                'interrogacion_count', 'porcentaje_mayusculas',
                'puntuacion_emocional', 'total_tokens', 'unique_types', 'ttr']

In [3]:
IN_COLAB = 'google.colab' in sys.modules

TOTAL_MEMORY = 0

try:
    import psutil
    TOTAL_MEMORY = psutil.virtual_memory().total
except ImportError:
    pass

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = Path('/content/drive/MyDrive/ML/proyecto_integrador')
else:
    BASE = Path('..')

DATA_DIR = BASE / 'data'
REPORTS_DIR = BASE / 'reports'
MODELS_DIR = BASE / 'models'
DATA_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

MLFLOW_TRACKING_URI = os.environ.get('MLFLOW_TRACKING_URI', 'https://humorous-trusting-domelike.ngrok-free.dev')
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment('f2a_modelado_clasico')
print(f"MLFLOW_TRACKING_URI: {MLFLOW_TRACKING_URI}")

print(f"=== Environment Info ===")
print(f"IN_COLAB: {IN_COLAB}")
if TOTAL_MEMORY:
    print(f"System RAM: {TOTAL_MEMORY / (1024**3):.1f} GB")
print(f"BASE: {BASE}")
print(f"DATA_DIR: {DATA_DIR}")

Mounted at /content/drive
MLFLOW_TRACKING_URI: https://humorous-trusting-domelike.ngrok-free.dev
=== Environment Info ===
IN_COLAB: True
System RAM: 12.7 GB
BASE: /content/drive/MyDrive/ML/proyecto_integrador
DATA_DIR: /content/drive/MyDrive/ML/proyecto_integrador/data


## Carga de Datos

Dataset canónico generado por `f1_eda_definitivo.ipynb`: 2.5M reseñas balanceadas.
La columna `target_class` contiene los labels: Negativo (1-2 estrellas), Neutro (3), Positivo (4-5).
Se mapean a valores numéricos 0/1/2 con LabelEncoder para sklearn.

In [4]:
print('Cargando dataset canonico desde f1_eda_definitivo...')
CANONICAL_PATH = DATA_DIR / 'office_products_balanced.parquet'

try:
    df = pd.read_parquet(CANONICAL_PATH, columns=['text', 'rating', 'target_class'] + FEATURE_COLS)
except FileNotFoundError:
    print(f'[ERROR] Dataset canonico no encontrado en {CANONICAL_PATH}')
    print('Ejecute primero f1_eda_definitivo.ipynb en Colab para generarlo.')
    raise
except Exception as e:
    print(f'[ERROR] No se pudo cargar el dataset: {e}')
    raise

label_encoder = LabelEncoder()
df['target'] = label_encoder.fit_transform(df['target_class'])
print('Mapeo de clases:', dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_))))

# word_count filter: purgar reseñas telegraficas sin peso semantico
df['word_count'] = df['text'].astype(str).str.split().str.len()
df = df[df['word_count'] >= 5]
print(f'Registros tras filtro word_count >= 5: {len(df):,}')

print('\nDistribucion de target (0=Negativo, 1=Neutro, 2=Positivo):')
print(df['target'].value_counts().sort_index())

SUBSAMPLE_SIZE = 1_000_000
print(f'\nSubmuestreo estratificado a {SUBSAMPLE_SIZE:,} filas...')
df, _ = train_test_split(
    df, train_size=SUBSAMPLE_SIZE, random_state=RANDOM_SEED, stratify=df['target']
)
print(f'Dataset reducido a {len(df):,} filas')
print(df['target'].value_counts().sort_index())

Cargando dataset canonico desde f1_eda_definitivo...
Mapeo de clases: {'Negativo': np.int64(0), 'Neutro': np.int64(1), 'Positivo': np.int64(2)}
Registros tras filtro word_count >= 5: 2,498,294

Distribucion de target (0=Negativo, 1=Neutro, 2=Positivo):
target
0    999652
1    499847
2    998795
Name: count, dtype: int64

Submuestreo estratificado a 1,000,000 filas...
Dataset reducido a 1,000,000 filas
target
0    400134
1    200075
2    399791
Name: count, dtype: int64


## División Train/Test

Se divide ANTES de vectorizar para prevenir Data Leakage: el vocabulario del conjunto de prueba no debe contaminar el entrenamiento.

In [5]:
print('Dividiendo datos en Train y Test (80/20)...')
X_train_text, X_test_text, X_train_feats, X_test_feats, y_train, y_test = train_test_split(
    df['text'], df[FEATURE_COLS], df['target'],
    test_size=0.20, random_state=RANDOM_SEED, stratify=df['target']
)

del df
gc.collect()

Dividiendo datos en Train y Test (80/20)...


17

## Vectorización TF-IDF + Features Engineered

TF-IDF con max 10K features + 9 features lingüísticas del EDA concatenadas.

In [6]:
print('Entrenando TF-IDF Vectorizer...')
vectorizer = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 2),
    stop_words='english',
    min_df=5
)

X_train_tfidf = vectorizer.fit_transform(X_train_text.astype(str))
X_test_tfidf = vectorizer.transform(X_test_text.astype(str))

print('Concatenando TF-IDF con features engineered...')
X_train = hstack([X_train_tfidf, csr_matrix(X_train_feats.values)])
X_test = hstack([X_test_tfidf, csr_matrix(X_test_feats.values)])

print(f'Matriz combinada: {X_train.shape}')
del X_train_tfidf, X_test_tfidf
gc.collect()

joblib.dump(vectorizer, MODELS_DIR / 'tfidf_vectorizer.joblib')
print(f'Vectorizer guardado en {MODELS_DIR / "tfidf_vectorizer.joblib"}')

Entrenando TF-IDF Vectorizer...
Concatenando TF-IDF con features engineered...
Matriz combinada: (800000, 10009)
Vectorizer guardado en /content/drive/MyDrive/ML/proyecto_integrador/models/tfidf_vectorizer.joblib


## Modelos Base

### Regresión Logística

In [7]:
print('Entrenando Regresion Logistica (Baseline)...')
import time as _t
_t0 = _t.time()
log_model = LogisticRegression(max_iter=2000, C=0.5, class_weight='balanced', random_state=RANDOM_SEED, n_jobs=-1)
log_model.fit(X_train, y_train)
logreg_time = _t.time() - _t0
y_pred_log = log_model.predict(X_test)

joblib.dump(log_model, MODELS_DIR / 'logreg_model.joblib')

with mlflow.start_run(run_name='logreg_baseline', nested=True):
    mlflow.set_tag('model_type', 'logistic_regression')
    mlflow.log_param('C', 0.5)
    mlflow.log_param('max_iter', 2000)
    mlflow.log_param('feature_count', X_train.shape[1])
    mlflow.log_metric('f1_macro', f1_score(y_test, y_pred_log, average='macro'))
    mlflow.log_metric('training_time_sec', logreg_time)

print(f'LogReg: {logreg_time:.1f}s | F1-macro: {f1_score(y_test, y_pred_log, average="macro"):.4f}')
print(f'Modelo LogReg guardado en {MODELS_DIR / "logreg_model.joblib"}')

Entrenando Regresion Logistica (Baseline)...
🏃 View run logreg_baseline at: https://humorous-trusting-domelike.ngrok-free.dev/#/experiments/6/runs/d24f68e79648451ca7a64d095c82cf83
🧪 View experiment at: https://humorous-trusting-domelike.ngrok-free.dev/#/experiments/6
Modelo LogReg guardado en /content/drive/MyDrive/ML/proyecto_integrador/models/logreg_model.joblib


### Random Forest

Limitado con `max_samples=0.05` y `max_depth=30` para controlar tiempo en CPU.

In [8]:
print('Entrenando Random Forest...')
_t0 = _t.time()
rf_model = RandomForestClassifier(
    n_estimators=100, max_samples=0.05, max_depth=30,
    class_weight='balanced', random_state=RANDOM_SEED, n_jobs=-1
)
rf_model.fit(X_train, y_train)
rf_time = _t.time() - _t0
y_pred_rf = rf_model.predict(X_test)

joblib.dump(rf_model, MODELS_DIR / 'rf_model.joblib')

with mlflow.start_run(run_name='random_forest', nested=True):
    mlflow.set_tag('model_type', 'random_forest')
    mlflow.log_param('n_estimators', 100)
    mlflow.log_param('max_samples', 0.05)
    mlflow.log_param('max_depth', 30)
    mlflow.log_metric('f1_macro', f1_score(y_test, y_pred_rf, average='macro'))
    mlflow.log_metric('training_time_sec', rf_time)

print(f'RF: {rf_time:.1f}s | F1-macro: {f1_score(y_test, y_pred_rf, average="macro"):.4f}')
print(f'Modelo RF guardado en {MODELS_DIR / "rf_model.joblib"}')

Entrenando Random Forest...
🏃 View run random_forest at: https://humorous-trusting-domelike.ngrok-free.dev/#/experiments/6/runs/380b3e7c7c344b91ab4a9cf85ef611c5
🧪 View experiment at: https://humorous-trusting-domelike.ngrok-free.dev/#/experiments/6
Modelo RF guardado en /content/drive/MyDrive/ML/proyecto_integrador/models/rf_model.joblib


### LightGBM

Tuneado con `num_leaves=64`, `learning_rate=0.05`, `min_child_samples=20`.

In [9]:
print('Entrenando LightGBM...')
_t0 = _t.time()
lgb_model = lgb.LGBMClassifier(
    class_weight='balanced', random_state=RANDOM_SEED,
    n_jobs=-1, verbose=-1,
    num_leaves=64, learning_rate=0.05, min_child_samples=20
)
lgb_model.fit(X_train, y_train)
lgb_time = _t.time() - _t0
y_pred_lgb = lgb_model.predict(X_test)

joblib.dump(lgb_model, MODELS_DIR / 'lgb_model.joblib')

with mlflow.start_run(run_name='lightgbm', nested=True):
    mlflow.set_tag('model_type', 'lightgbm')
    mlflow.log_param('num_leaves', 64)
    mlflow.log_param('learning_rate', 0.05)
    mlflow.log_metric('f1_macro', f1_score(y_test, y_pred_lgb, average='macro'))
    mlflow.log_metric('training_time_sec', lgb_time)

print(f'LGB: {lgb_time:.1f}s | F1-macro: {f1_score(y_test, y_pred_lgb, average="macro"):.4f}')
print(f'Modelo LightGBM guardado en {MODELS_DIR / "lgb_model.joblib"}')

Entrenando LightGBM...


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


🏃 View run lightgbm at: https://humorous-trusting-domelike.ngrok-free.dev/#/experiments/6/runs/41ac3bdfb2954edf89da4e785c54253e
🧪 View experiment at: https://humorous-trusting-domelike.ngrok-free.dev/#/experiments/6
Modelo LightGBM guardado en /content/drive/MyDrive/ML/proyecto_integrador/models/lgb_model.joblib


### XGBoost (CPU)

Optimizado con `tree_method='hist'` en CPU.

In [10]:
gc.collect()

print('Entrenando XGBoost...')

classes = np.unique(y_train)
cw = compute_class_weight('balanced', classes=classes, y=y_train)
class_weights_dict = {int(c): float(w) for c, w in zip(classes, cw)}

_t0 = _t.time()
xgb_model = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    objective='multi:softprob',
    num_class=3,
    random_state=RANDOM_SEED,
    eval_metric='mlogloss',
    tree_method='hist',
)

sample_weights = np.array([class_weights_dict[y] for y in y_train])
xgb_model.fit(X_train, y_train, sample_weight=sample_weights)
xgb_time = _t.time() - _t0
y_pred_xgb = xgb_model.predict(X_test)

joblib.dump(xgb_model, MODELS_DIR / 'xgb_model.joblib')

with mlflow.start_run(run_name='xgboost', nested=True):
    mlflow.set_tag('model_type', 'xgboost')
    mlflow.log_param('n_estimators', 300)
    mlflow.log_param('max_depth', 5)
    mlflow.log_param('learning_rate', 0.05)
    mlflow.log_param('tree_method', 'hist')
    mlflow.log_metric('f1_macro', f1_score(y_test, y_pred_xgb, average='macro'))
    mlflow.log_metric('training_time_sec', xgb_time)

print(f'XGB: {xgb_time:.1f}s | F1-macro: {f1_score(y_test, y_pred_xgb, average="macro"):.4f}')
print(f'Modelo XGBoost guardado en {MODELS_DIR / "xgb_model.joblib"}')

Entrenando XGBoost...
🏃 View run xgboost at: https://humorous-trusting-domelike.ngrok-free.dev/#/experiments/6/runs/bc9a0c44c10f443e871805f8795e94a9
🧪 View experiment at: https://humorous-trusting-domelike.ngrok-free.dev/#/experiments/6
Entrenamiento completado.
Modelo XGBoost guardado en /content/drive/MyDrive/ML/proyecto_integrador/models/xgb_model.joblib


## AutoML Benchmark

LazyPredict ejecuta 27 clasificadores sobre una muestra estratificada de 20K train / 5K test.

In [ ]:
from lazypredict.Supervised import LazyClassifier

print('Muestreando 10K train / 3K test para AutoML benchmark...')
X_train_sample, _, y_train_sample, _ = train_test_split(
    X_train, y_train, train_size=10000, random_state=RANDOM_SEED, stratify=y_train
)
X_test_sample, _, y_test_sample, _ = train_test_split(
    X_test, y_test, test_size=3000, random_state=RANDOM_SEED, stratify=y_test
)

print('Iniciando Benchmark AutoML (LazyPredict)...')
print('Esto puede tomar varios minutos: evalua 27 clasificadores sobre la muestra.')
clf = LazyClassifier(verbose=0, ignore_warnings=True, custom_metric=None)
# Convert sparse matrices to dense arrays for LazyPredict, as it seems to have issues with len() on sparse inputs
models_df, predictions = clf.fit(X_train_sample.toarray(), X_test_sample.toarray(), y_train_sample, y_test_sample)

print('\nTop 10 modelos segun LazyPredict:')
print(models_df.head(10).to_string())

lazy_path = REPORTS_DIR / 'lazypredict_results.csv'
models_df.to_csv(lazy_path)
print(f'Resultados completos guardados en {lazy_path}')

Muestreando 20K train / 5K test para AutoML benchmark...


2026/06/13 01:45:43 INFO mlflow.tracking.fluent: Autologging successfully enabled for lightgbm.
2026/06/13 01:45:43 INFO mlflow.tracking.fluent: Autologging successfully enabled for sklearn.
2026/06/13 01:45:43 INFO mlflow.tracking.fluent: Autologging successfully enabled for statsmodels.
2026/06/13 01:45:43 INFO mlflow.tracking.fluent: Autologging successfully enabled for xgboost.
2026/06/13 01:45:43 WARNING mlflow.spark: With Pyspark >= 3.2, PYSPARK_PIN_THREAD environment variable must be set to false for Spark datasource autologging to work.
2026/06/13 01:45:43 INFO mlflow.tracking.fluent: Autologging successfully enabled for pyspark.


Iniciando Benchmark AutoML (LazyPredict)...
Esto puede tomar varios minutos: evalua 27 clasificadores sobre la muestra.


## Evaluación Comparativa

Métrica rectora: F1-macro sobre la partición de test (20% = 500K muestras). Se exportan métricas a JSON y MLflow.

In [ ]:
from matplotlib.colors import Normalize
from sklearn.metrics import ConfusionMatrixDisplay

FIGURES_DIR = REPORTS_DIR / 'figures' / 'fase2'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# Subsample test set for faster plotting (20K)
EVAL_SAMPLE = 20000
print(f'Evaluando sobre muestra de {EVAL_SAMPLE:,} registros de test...')
_, y_test_eval, _, y_pred_log_eval = train_test_split(
    y_test, y_pred_log, train_size=EVAL_SAMPLE, random_state=RANDOM_SEED, stratify=y_test
)
_, _, _, y_pred_rf_eval = train_test_split(
    y_test, y_pred_rf, train_size=EVAL_SAMPLE, random_state=RANDOM_SEED, stratify=y_test
)
_, _, _, y_pred_lgb_eval = train_test_split(
    y_test, y_pred_lgb, train_size=EVAL_SAMPLE, random_state=RANDOM_SEED, stratify=y_test
)
_, _, _, y_pred_xgb_eval = train_test_split(
    y_test, y_pred_xgb, train_size=EVAL_SAMPLE, random_state=RANDOM_SEED, stratify=y_test
)

models = {
    'Logistic Regression': y_pred_log_eval,
    'Random Forest': y_pred_rf_eval,
    'LightGBM': y_pred_lgb_eval,
    'XGBoost': y_pred_xgb_eval,
}
model_times = {
    'Logistic Regression': logreg_time,
    'Random Forest': rf_time,
    'LightGBM': lgb_time,
    'XGBoost': xgb_time,
}
model_colors = ['#4C72B0', '#DD8452', '#55A868', '#C44E52']
class_labels = ['Negativo', 'Neutro', 'Positivo']

results = []
model_names = list(models.keys())
preds_list = list(models.values())

# ---- Compute metrics ----
f1_macros = []
accuracies = []
all_cm = []
all_per_class = []

for name, preds in models.items():
    f1_macros.append(f1_score(y_test_eval, preds, average='macro'))
    accuracies.append(accuracy_score(y_test_eval, preds))
    cm = confusion_matrix(y_test_eval, preds)
    all_cm.append(cm)
    p_class, r_class, f1_class, _ = precision_recall_fscore_support(
        y_test_eval, preds, labels=[0, 1, 2]
    )
    all_per_class.append({'precision': p_class, 'recall': r_class, 'f1': f1_class})
    results.append({
        'model_name': name,
        'model_type': name.lower().replace(' ', '_'),
        'f1_macro': round(f1_macros[-1], 4),
        'accuracy': round(accuracies[-1], 4),
        'training_time_sec': round(model_times[name], 1),
        'confusion_matrix': cm.tolist(),
        'per_class': {
            cl: {
                'precision': round(p_class[i], 4),
                'recall': round(r_class[i], 4),
                'f1': round(f1_class[i], 4)
            } for i, cl in enumerate(class_labels)
        }
    })

best_idx = int(np.argmax(f1_macros))

# ============================================================
# FIG 1: 2x2 Confusion Matrices
# ============================================================
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
for idx, (name, cm) in enumerate(zip(model_names, all_cm)):
    ax = axes[idx // 2][idx % 2]
    disp = ConfusionMatrixDisplay(cm, display_labels=class_labels)
    disp.plot(ax=ax, cmap='Blues', colorbar=False, values_format='d')
    ax.set_title(f'{name}\nF1-macro: {f1_macros[idx]:.4f}', fontsize=11)
    ax.grid(False)
plt.suptitle('Matrices de Confusion - Modelos Clasicos (F2A)', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'confusion_matrices_2x2.png', dpi=150, bbox_inches='tight')
plt.show()

# ============================================================
# FIG 2: F1-macro Comparison (horizontal bar)
# ============================================================
fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(model_names, f1_macros, color=model_colors, edgecolor='white', height=0.6)
for bar, val in zip(bars, f1_macros):
    ax.text(val + 0.005, bar.get_y() + bar.get_height() / 2,
            f'{val:.4f}', va='center', fontsize=10)
ax.set_xlabel('F1-macro')
ax.set_title('Comparacion de F1-macro por Modelo')
ax.set_xlim(0, max(f1_macros) + 0.05)
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'f1_macro_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

# ============================================================
# FIG 3: Per-class Precision / Recall / F1 (grouped bars)
# ============================================================
fig, axes = plt.subplots(1, 4, figsize=(16, 4.5))
metrics_names = ['precision', 'recall', 'f1']
metrics_labels = ['Precision', 'Recall', 'F1-score']
bar_width = 0.25
x = np.arange(len(class_labels))
for idx, name in enumerate(model_names):
    ax = axes[idx]
    pc = all_per_class[idx]
    for j, (mkey, mlabel) in enumerate(zip(metrics_names, metrics_labels)):
        offset = (j - 1) * bar_width
        bars = ax.bar(x + offset, pc[mkey], bar_width, label=mlabel,
                       color=['#4C72B0', '#55A868', '#C44E52'][j], alpha=0.85)
    ax.set_xticks(x)
    ax.set_xticklabels(class_labels, fontsize=9)
    ax.set_ylim(0, 1.0)
    ax.set_title(name, fontsize=10)
    ax.legend(fontsize=7, loc='lower right')
    ax.grid(axis='y', alpha=0.3)
axes[0].set_ylabel('Score')
plt.suptitle('Metricas por Clase', fontsize=13)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'per_class_metrics.png', dpi=150, bbox_inches='tight')
plt.show()

# ============================================================
# FIG 4: Training Time Comparison
# ============================================================
fig, ax = plt.subplots(figsize=(10, 4))
times = [model_times[n] for n in model_names]
bars = ax.barh(model_names, times, color=model_colors, edgecolor='white', height=0.6)
for bar, val in zip(bars, times):
    ax.text(val + 5, bar.get_y() + bar.get_height() / 2,
            f'{val:.0f}s', va='center', fontsize=10)
ax.set_xlabel('Tiempo de Entrenamiento (segundos)')
ax.set_title('Tiempo de Entrenamiento por Modelo')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'training_times.png', dpi=150, bbox_inches='tight')
plt.show()

# ============================================================
# FIG 5: Normalized Confusion Matrix (error analysis)
# ============================================================
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
for idx, (name, cm) in enumerate(zip(model_names, all_cm)):
    ax = axes[idx // 2][idx % 2]
    cm_norm = cm.astype('float') / cm.sum(axis=1, keepdims=True)
    im = ax.imshow(cm_norm, cmap='RdYlGn', aspect='auto', vmin=0, vmax=1)
    for i in range(3):
        for j in range(3):
            val = cm_norm[i, j]
            color = 'white' if val > 0.5 else 'black'
            ax.text(j, i, f'{val:.0%}\n({cm[i,j]})', ha='center', va='center',
                    color=color, fontsize=9)
    ax.set_xticks(range(3))
    ax.set_yticks(range(3))
    ax.set_xticklabels(class_labels)
    ax.set_yticklabels(class_labels)
    ax.set_title(f'{name} - F1: {f1_macros[idx]:.4f}', fontsize=11)
    ax.set_ylabel('Real')
    ax.set_xlabel('Predicho')
    ax.grid(False)
plt.suptitle('Matrices de Confusion Normalizadas - Analisis de Error', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'confusion_normalized.png', dpi=150, bbox_inches='tight')
plt.show()

# ---- Print summary ----
print('\\n' + '='*60)
print('RESUMEN - F2A Modelado Clasico')
print('='*60)
print(f'{"Modelo":25s} {"F1-macro":>10s} {"Acc":>8s} {"Tiempo":>8s}')
print('-'*55)
for i, name in enumerate(model_names):
    star = ' *' if i == best_idx else '  '
    print(f'{name + star:25s} {f1_macros[i]:10.4f} {accuracies[i]:8.4f} {model_times[name]:7.0f}s')
print(f'\\nMejor modelo: {model_names[best_idx]} (F1-macro: {f1_macros[best_idx]:.4f})')

# ---- Export metrics ----
report_path = REPORTS_DIR / 'metrics_fase2.json'
try:
    with open(report_path, 'w') as f:
        json.dump(results, f, indent=2, ensure_ascii=False)
    print(f'[METRICAS] Exportadas a {report_path}')
except Exception as e:
    print(f'[ERROR] No se pudo exportar metricas: {e}')

# ---- MLflow ----
with mlflow.start_run(run_name='evaluation_comparativa'):
    mlflow.set_tag('phase', '2a')
    mlflow.set_tag('dataset_size', '1M_subsample')
    mlflow.log_param('eval_sample_size', EVAL_SAMPLE)
    for i, name in enumerate(model_names):
        mlflow.log_metric(f'{name.lower().replace(" ", "_")}_f1_macro', f1_macros[i])
        mlflow.log_metric(f'{name.lower().replace(" ", "_")}_accuracy', accuracies[i])
        mlflow.log_metric(f'{name.lower().replace(" ", "_")}_time_sec', model_times[name])
    mlflow.log_metric('best_f1_macro', f1_macros[best_idx])
    mlflow.log_text('best_model', model_names[best_idx])
    mlflow.log_artifact(str(FIGURES_DIR / 'confusion_matrices_2x2.png'))
    mlflow.log_artifact(str(FIGURES_DIR / 'f1_macro_comparison.png'))
    mlflow.log_artifact(str(FIGURES_DIR / 'per_class_metrics.png'))
    mlflow.log_artifact(str(FIGURES_DIR / 'training_times.png'))